# Sidecar to Ambient Upgrade

A follow-along migration of a petstore app from Istio **sidecar** mode to **ambient**, one namespace at a
time, on the Solo distribution of Istio, on one kind cluster. One script sets the cluster up; everything after
that is plain YAML you apply and read, so you can talk through every change.

The shape of it: turn the mesh ambient with one field, migrate an **L4-only** namespace with no waypoint,
migrate an **L7** namespace behind a waypoint (with its DestinationRule and VirtualService still working),
route the **mixed fleet** (sidecar + ingress) through the waypoint, optionally move the canary to
**HTTPRoute**, and **roll back** with a single label. A fortio load generator proves zero downtime at each cut.

## Before you start

Kernel: this notebook runs shell (Cursor: Select Kernel → Jupyter Kernel → Bash;
`pip install bash_kernel && python -m bash_kernel.install` if you need it).

You need Docker, `kind`, `kubectl`, `helm`, `gcloud` (authenticated, for the Solo images) and a
`SOLO_ISTIO_LICENSE_KEY`. Set that up in the next cell. The optional appendix also uses the `gloo` CLI.

In [ ]:
export SECRETS_FILE="${SECRETS_FILE:-$HOME/secrets/secrets-envs.sh}"   # must export SOLO_ISTIO_LICENSE_KEY
export CTX=kind-ambient-migration
kc() { kubectl --context "$CTX" "$@"; }        # persists across cells (one Bash session)
echo "context: $CTX"

## 1. Set up the cluster

The one script: a kind cluster, the Gateway API CRDs, the Gloo Operator, a `ServiceMeshController` in
**sidecar** mode (Solo Istio), and the Istio ingress gateway. It leaves the mesh in sidecar mode with nothing
else deployed, ready for the walkthrough.

In [ ]:
./scripts/setup-cluster.sh

Sidecar mode: there is an `istio-cni` DaemonSet (it does traffic redirection) but **no ztunnel** yet. istiod carries `ENABLE_WAYPOINT_INTEROP=true`, the Solo default that makes the mixed-fleet step work later.

In [ ]:
kc -n istio-system get deploy,ds
kc -n istio-system get deploy istiod-gloo \
  -o jsonpath='{range .spec.template.spec.containers[0].env[*]}{.name}={.value}{"\n"}{end}' | grep WAYPOINT_INTEROP

## 2. Deploy the petstore app (sidecar mode)

Three namespaces, all starting with `istio-injection=enabled`:

- **petstore** (L7): a `catalog` Service fronting v1 and v2 Deployments.
- **petstore-data** (L4 only): Redis over plain TCP.
- **petstore-legacy**: a `checkout` client and a `fortio` load generator we keep on sidecars throughout, so
  they double as mixed-fleet callers.

In [ ]:
kc apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata: { name: petstore, labels: { istio-injection: enabled } }
---
apiVersion: v1
kind: Namespace
metadata: { name: petstore-data, labels: { istio-injection: enabled } }
---
apiVersion: v1
kind: Namespace
metadata: { name: petstore-legacy, labels: { istio-injection: enabled } }
EOF

**catalog** — one Service (`app: catalog`) fronting two versioned Deployments. Each replica is http-echo returning its version, so a caller can see which one served it.

In [ ]:
kc apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata: { name: catalog, namespace: petstore }
---
apiVersion: v1
kind: Service
metadata: { name: catalog, namespace: petstore, labels: { app: catalog } }
spec:
  selector: { app: catalog }
  ports: [ { name: http, port: 80, targetPort: 5678 } ]   # port name http* → L7-capable
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: catalog-v1, namespace: petstore }
spec:
  replicas: 2
  selector: { matchLabels: { app: catalog, version: v1 } }
  template:
    metadata: { labels: { app: catalog, version: v1 } }
    spec:
      serviceAccountName: catalog
      containers:
        - name: catalog
          image: hashicorp/http-echo:1.0
          args: ["-listen=:5678", "-text={\"app\":\"catalog\",\"version\":\"v1\"}"]
          ports: [ { containerPort: 5678 } ]
          resources: { requests: { cpu: 10m, memory: 16Mi }, limits: { memory: 64Mi } }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: catalog-v2, namespace: petstore }
spec:
  replicas: 2
  selector: { matchLabels: { app: catalog, version: v2 } }
  template:
    metadata: { labels: { app: catalog, version: v2 } }
    spec:
      serviceAccountName: catalog
      containers:
        - name: catalog
          image: hashicorp/http-echo:1.0
          args: ["-listen=:5678", "-text={\"app\":\"catalog\",\"version\":\"v2\"}"]
          ports: [ { containerPort: 5678 } ]
          resources: { requests: { cpu: 10m, memory: 16Mi }, limits: { memory: 64Mi } }
EOF

**petstore-data** — Redis (the L4 target) plus a `data-client` in petstore that pings it under the `catalog` identity.

In [ ]:
kc apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata: { name: redis, namespace: petstore-data }
---
apiVersion: v1
kind: Service
metadata: { name: redis, namespace: petstore-data, labels: { app: redis } }
spec:
  selector: { app: redis }
  ports: [ { name: tcp-redis, port: 6379, targetPort: 6379 } ]   # tcp-* → L4
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: redis, namespace: petstore-data }
spec:
  replicas: 1
  selector: { matchLabels: { app: redis } }
  template:
    metadata: { labels: { app: redis } }
    spec:
      serviceAccountName: redis
      containers:
        - name: redis
          image: redis:7-alpine
          args: ["--save","","--appendonly","no"]
          ports: [ { containerPort: 6379 } ]
          resources: { requests: { cpu: 10m, memory: 16Mi }, limits: { memory: 64Mi } }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: data-client, namespace: petstore }
spec:
  replicas: 1
  selector: { matchLabels: { app: data-client } }
  template:
    metadata: { labels: { app: data-client } }
    spec:
      serviceAccountName: catalog          # allowed principal for the Redis L4 rule
      containers:
        - name: client
          image: redis:7-alpine
          command: ["/bin/sh","-c"]
          args: ["while true; do redis-cli -h redis.petstore-data ping >/dev/null 2>&1 && echo \"$(date +%T) PONG\" || echo \"$(date +%T) FAIL\"; sleep 5; done"]
          resources: { requests: { cpu: 10m, memory: 16Mi }, limits: { memory: 64Mi } }
EOF

**petstore-legacy** — `checkout` (a sidecar HTTP client) and `fortio` (the zero-downtime load generator). Both stay on sidecars the whole way.

In [ ]:
kc apply -f - <<'EOF'
apiVersion: v1
kind: ServiceAccount
metadata: { name: checkout, namespace: petstore-legacy }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: checkout, namespace: petstore-legacy }
spec:
  replicas: 1
  selector: { matchLabels: { app: checkout } }
  template:
    metadata: { labels: { app: checkout } }
    spec:
      serviceAccountName: checkout
      containers:
        - name: checkout
          image: curlimages/curl:8.10.1
          command: ["/bin/sh","-c","sleep infinity"]
          resources: { requests: { cpu: 10m, memory: 16Mi }, limits: { memory: 64Mi } }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: fortio, namespace: petstore-legacy }
spec:
  replicas: 1
  selector: { matchLabels: { app: fortio } }
  template:
    metadata: { labels: { app: fortio } }
    spec:
      containers:
        - name: fortio
          image: fortio/fortio:latest
          command: ["fortio","server"]
          ports: [ { containerPort: 8080 } ]
          resources: { requests: { cpu: 10m, memory: 32Mi }, limits: { memory: 128Mi } }
EOF
for d in "petstore catalog-v1" "petstore catalog-v2" "petstore data-client" "petstore-data redis" "petstore-legacy checkout" "petstore-legacy fortio"; do
  set -- $d; kc -n "$1" rollout status deploy/"$2" --timeout=120s; done

## 3. Sidecar-era policies + the ingress route

STRICT mTLS mesh-wide; the catalog DestinationRule + VirtualService (subset canary, retries, timeout); an L4
rule on Redis (identity only); an L7 rule on catalog (GET/HEAD only, `selector`-based); and the classic Istio
`Gateway` + `VirtualService` ingress route.

In [ ]:
kc apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: PeerAuthentication
metadata: { name: default, namespace: istio-system }
spec: { mtls: { mode: STRICT } }
---
apiVersion: networking.istio.io/v1
kind: DestinationRule
metadata: { name: catalog, namespace: petstore }
spec:
  host: catalog.petstore.svc.cluster.local
  trafficPolicy:
    connectionPool: { http: { http1MaxPendingRequests: 100, maxRequestsPerConnection: 10 } }
    outlierDetection: { consecutive5xxErrors: 5, interval: 10s, baseEjectionTime: 30s }
  subsets:
    - { name: v1, labels: { version: v1 } }
    - { name: v2, labels: { version: v2 } }
---
apiVersion: networking.istio.io/v1
kind: VirtualService
metadata: { name: catalog, namespace: petstore }
spec:
  hosts: [ catalog.petstore.svc.cluster.local ]
  http:
    - timeout: 3s
      retries: { attempts: 3, perTryTimeout: 1s, retryOn: "5xx,reset,connect-failure" }
      route:
        - { destination: { host: catalog.petstore.svc.cluster.local, subset: v1 }, weight: 100 }
        - { destination: { host: catalog.petstore.svc.cluster.local, subset: v2 }, weight: 0 }
---
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: redis-allow-catalog, namespace: petstore-data }
spec:
  selector: { matchLabels: { app: redis } }
  action: ALLOW
  rules: [ { from: [ { source: { principals: ["cluster.local/ns/petstore/sa/catalog"] } } ] } ]
---
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: catalog-get-only, namespace: petstore }
spec:
  selector: { matchLabels: { app: catalog } }
  action: ALLOW
  rules: [ { to: [ { operation: { methods: ["GET","HEAD"] } } ] } ]
---
apiVersion: networking.istio.io/v1
kind: Gateway
metadata: { name: petstore-gateway, namespace: istio-ingress }
spec:
  selector: { istio: ingressgateway }
  servers:
    - port: { number: 80, name: http, protocol: HTTP }
      hosts: [ "petstore.local" ]
---
apiVersion: networking.istio.io/v1
kind: VirtualService
metadata: { name: petstore-ingress, namespace: istio-ingress }
spec:
  hosts: [ "petstore.local" ]
  gateways: [ petstore-gateway ]
  http:
    - route: [ { destination: { host: catalog.petstore.svc.cluster.local, port: { number: 80 } } } ]
EOF

**Baseline (sidecar mode).** Every pod is 2/2 (app + sidecar), the canary is 100% v1, GET is allowed and DELETE denied, the L4 rule lets the catalog identity into Redis but no one else, and fortio is clean.

In [ ]:
echo "=== pods 2/2 ==="; kc get pods -n petstore | head
echo "=== canary (in-mesh): 100% v1 ==="
kc -n petstore-legacy exec deploy/checkout -c checkout -- sh -c 'for i in $(seq 1 12); do curl -s http://catalog.petstore/; echo; done' | grep -o 'v[12]' | sort | uniq -c
echo "=== L7 authz: GET 200, DELETE 403 ==="
kc -n petstore-legacy exec deploy/checkout -c checkout -- sh -c 'echo -n "GET->"; curl -s -o /dev/null -w "%{http_code}\n" http://catalog.petstore/; echo -n "DELETE->"; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE http://catalog.petstore/'
echo "=== L4 allow (catalog identity) ==="
kc -n petstore exec deploy/data-client -c client -- sh -c 'timeout 5 redis-cli -h redis.petstore-data ping'
echo "=== ingress (Host: petstore.local) ==="
curl -s -o /dev/null -w "ingress=%{http_code}\n" -H 'Host: petstore.local' http://localhost:18080/
echo "=== fortio baseline ==="
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 500 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

### The one decision per namespace: L4 or L7

ztunnel (no waypoint) does mTLS, identity/namespace/IP/port authorization, TCP telemetry, and on Solo also
HTTP telemetry and outlier detection. You need a **waypoint** only for L7 rules: HTTP method/path/header
authz, JWT, or VirtualService/HTTPRoute behaviour. So **petstore-data** (identity TCP rule) needs no waypoint;
**petstore** (HTTP method rule + VS routing) needs one before it is enrolled. Enrol an L7 namespace without a
waypoint and ztunnel fails safe: it denies all traffic to that workload.

## 4. Turn on ambient (one field)

Flip the `ServiceMeshController` to `dataplaneMode: Ambient`. The Gloo Operator adds the `istio-cni` and
`ztunnel` DaemonSets and switches istiod into ambient mode. Running sidecar workloads are untouched, so the
flip is zero-downtime and nothing is enrolled yet.

In [ ]:
kc patch servicemeshcontroller managed-istio -n gloo-system --type=merge -p '{"spec":{"dataplaneMode":"Ambient"}}'
until kc -n istio-system get ds ztunnel >/dev/null 2>&1; do sleep 5; done
kc -n istio-system rollout status ds/ztunnel --timeout=150s
echo "=== ztunnel + istio-cni, one per node ==="; kc -n istio-system get ds
echo "=== sidecar apps untouched; fortio clean across the flip ==="
kc get pods -n petstore | grep catalog
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 600 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

## 5. Migrate the L4-only namespace (no waypoint)

`petstore-data` has only an identity rule, so ztunnel does all of it. Enrol it, stop injection, restart.
Redis returns 1/1 with no sidecar and the same L4 rule is enforced, now by ztunnel.

In [ ]:
kc label ns petstore-data istio.io/dataplane-mode=ambient istio-injection- --overwrite
kc -n petstore-data rollout restart deploy/redis
kc -n petstore-data rollout status deploy/redis --timeout=120s
echo "=== redis 1/1 (no sidecar), no waypoint in the namespace ==="
kc get pods -n petstore-data; kc -n petstore-data get gateway 2>/dev/null || echo "  (no waypoint)"
echo "=== allowed (catalog identity) ==="
kc -n petstore exec deploy/data-client -c client -- sh -c 'timeout 5 redis-cli -h redis.petstore-data ping'
echo "=== denied (checkout identity) ==="
kc -n petstore-legacy delete pod l4test --ignore-not-found >/dev/null 2>&1
kc -n petstore-legacy run l4test --image=redis:7-alpine --overrides='{"spec":{"serviceAccountName":"checkout"}}' --restart=Never --command -- sleep 60 >/dev/null 2>&1
kc -n petstore-legacy wait --for=condition=Ready pod/l4test --timeout=40s >/dev/null 2>&1
kc -n petstore-legacy exec l4test -c l4test -- sh -c 'timeout 6 redis-cli -h redis.petstore-data ping 2>&1'
kc -n petstore-legacy delete pod l4test --ignore-not-found >/dev/null 2>&1

## 6. Migrate the L7 namespace (behind a waypoint)

Order matters. Deploy the waypoint first, translate the AuthorizationPolicy from `selector` to `targetRefs`,
bind catalog to the waypoint, then enrol and drop the old selector policy. The DestinationRule and
VirtualService keep working, now on the waypoint.

In [ ]:
# 1) the waypoint — a Gateway-API Gateway with the istio-waypoint class
kc apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata: { name: waypoint, namespace: petstore, labels: { istio.io/waypoint-for: service } }
spec:
  gatewayClassName: istio-waypoint
  listeners: [ { name: mesh, port: 15008, protocol: HBONE } ]
EOF
kc -n petstore wait --for=condition=Programmed gateway/waypoint --timeout=120s

In [ ]:
# 2) the L7 policy translated selector → targetRefs, and bind catalog to the waypoint
kc apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: catalog-get-only-waypoint, namespace: petstore }
spec:
  targetRefs: [ { kind: Service, group: "", name: catalog } ]
  action: ALLOW
  rules: [ { to: [ { operation: { methods: ["GET","HEAD"] } } ] } ]
EOF
kc -n petstore label service catalog istio.io/use-waypoint=waypoint --overwrite

In [ ]:
# 3) enrol petstore, drop the old selector policy, restart
kc label ns petstore istio.io/dataplane-mode=ambient istio-injection- --overwrite
kc -n petstore delete authorizationpolicy catalog-get-only --ignore-not-found
kc -n petstore rollout restart deploy/catalog-v1 deploy/catalog-v2 deploy/data-client
for d in catalog-v1 catalog-v2 data-client; do kc -n petstore rollout status deploy/$d --timeout=120s; done
kc get pods -n petstore | grep -E "catalog|data-client|waypoint"

The canary still routes at the waypoint, and it is a real split: shift the VirtualService to 50/50 and both versions appear. The still-sidecar `checkout` is enforced by the waypoint too (GET 200 / DELETE 403) — the Solo interop. fortio stays clean.

In [ ]:
echo "=== canary via waypoint: 100% v1 ==="
kc -n petstore exec deploy/data-client -c client -- sh -c 'for i in $(seq 1 20); do wget -qO- http://catalog.petstore/ 2>/dev/null; echo; done' | grep -o 'v[12]' | sort | uniq -c
echo "=== shift VS 50/50 — real split at the waypoint ==="
kc -n petstore patch virtualservice catalog --type=json -p='[{"op":"replace","path":"/spec/http/0/route/0/weight","value":50},{"op":"replace","path":"/spec/http/0/route/1/weight","value":50}]' >/dev/null; sleep 3
kc -n petstore exec deploy/data-client -c client -- sh -c 'for i in $(seq 1 40); do wget -qO- http://catalog.petstore/ 2>/dev/null; echo; done' | grep -o 'v[12]' | sort | uniq -c
kc -n petstore patch virtualservice catalog --type=json -p='[{"op":"replace","path":"/spec/http/0/route/0/weight","value":100},{"op":"replace","path":"/spec/http/0/route/1/weight","value":0}]' >/dev/null
echo "=== sidecar checkout enforced by the waypoint ==="
kc -n petstore-legacy exec deploy/checkout -c checkout -- sh -c 'echo -n "GET->"; curl -s -o /dev/null -w "%{http_code}\n" http://catalog.petstore/; echo -n "DELETE->"; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE http://catalog.petstore/'
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 600 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

## 7. The mixed fleet: route ingress through the waypoint

The in-mesh sidecar caller was routed through the waypoint automatically. The Istio ingress gateway needs one
label, `istio.io/ingress-use-waypoint`. Before it, the ingress bypasses the waypoint (DELETE allowed, canary
ignored); after it, the waypoint's authz and canary apply to north-south traffic too.

In [ ]:
echo "=== BEFORE: ingress bypasses the waypoint ==="
echo -n "ingress DELETE -> "; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE -H 'Host: petstore.local' http://localhost:18080/
echo -n "ingress versions -> "; for i in $(seq 1 20); do curl -s -H 'Host: petstore.local' http://localhost:18080/; echo; done | grep -o 'v[12]' | sort | uniq -c | tr '\n' ' '; echo
kc -n petstore label service catalog istio.io/ingress-use-waypoint=true --overwrite
sleep 5
echo "=== AFTER: ingress routed through the waypoint ==="
echo -n "ingress DELETE -> "; curl -s -o /dev/null -w "%{http_code}\n" -X DELETE -H 'Host: petstore.local' http://localhost:18080/
echo -n "ingress versions -> "; for i in $(seq 1 20); do curl -s -H 'Host: petstore.local' http://localhost:18080/; echo; done | grep -o 'v[12]' | sort | uniq -c | tr '\n' ' '; echo

## 8. Optional: move the canary from subsets to HTTPRoute

The durable version split. Give each version a real Service, split with an `HTTPRoute` parented on the
original catalog Service (callers keep the same hostname), keep the DestinationRule for its traffic policy,
retire the VirtualService. Shift `backendRef` weights to canary.

In [ ]:
kc apply -f - <<'EOF'
apiVersion: v1
kind: Service
metadata: { name: catalog-v1, namespace: petstore, labels: { app: catalog } }
spec:
  selector: { app: catalog, version: v1 }
  ports: [ { name: http, port: 80, targetPort: 5678 } ]
---
apiVersion: v1
kind: Service
metadata: { name: catalog-v2, namespace: petstore, labels: { app: catalog } }
spec:
  selector: { app: catalog, version: v2 }
  ports: [ { name: http, port: 80, targetPort: 5678 } ]
---
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata: { name: catalog, namespace: petstore }
spec:
  parentRefs: [ { group: "", kind: Service, name: catalog, port: 80 } ]
  rules:
    - backendRefs:
        - { name: catalog-v1, port: 80, weight: 100 }
        - { name: catalog-v2, port: 80, weight: 0 }
---
apiVersion: networking.istio.io/v1
kind: DestinationRule
metadata: { name: catalog, namespace: petstore }
spec:
  host: catalog.petstore.svc.cluster.local
  trafficPolicy:
    outlierDetection: { consecutive5xxErrors: 5, interval: 10s, baseEjectionTime: 30s }
EOF
kc -n petstore delete virtualservice catalog --ignore-not-found
echo "=== shift HTTPRoute to 30/70 ==="
kc -n petstore patch httproute catalog --type=json -p='[{"op":"replace","path":"/spec/rules/0/backendRefs/0/weight","value":30},{"op":"replace","path":"/spec/rules/0/backendRefs/1/weight","value":70}]' >/dev/null; sleep 3
kc -n petstore exec deploy/data-client -c client -- sh -c 'for i in $(seq 1 40); do wget -qO- http://catalog.petstore/ 2>/dev/null; echo; done' | grep -o 'v[12]' | sort | uniq -c
kc -n petstore patch httproute catalog --type=json -p='[{"op":"replace","path":"/spec/rules/0/backendRefs/0/weight","value":100},{"op":"replace","path":"/spec/rules/0/backendRefs/1/weight","value":0}]' >/dev/null
kc -n petstore-legacy exec deploy/fortio -c fortio -- fortio load -c 8 -n 600 -qps 0 -quiet http://catalog.petstore/ 2>&1 | grep "Code "

## 9. Rollback

The safety net. Any namespace goes back to sidecars with a single label flip. Shown on `petstore-data`; the
L4 rule stays enforced across the round trip.

In [ ]:
kc label ns petstore-data istio.io/dataplane-mode- istio-injection=enabled --overwrite
kc -n petstore-data rollout restart deploy/redis
kc -n petstore-data rollout status deploy/redis --timeout=120s
echo "=== redis back to 2/2 (sidecar); L4 rule still holds ==="
kc get pods -n petstore-data
kc -n petstore exec deploy/data-client -c client -- sh -c 'timeout 5 redis-cli -h redis.petstore-data ping'

## Appendix: the gloo CLI (estimate and migrate)

The `gloo` CLI plans the migration for you and changes nothing in the cluster. Install it, then `estimate`
writes a cost-estimate report and `migrate` reads the cluster and generates recommended YAML per phase.
(`meshctl experimental interop-check` runs the same interoperability checks from the Gloo Mesh CLI.)

In [ ]:
curl -sL https://storage.googleapis.com/gloo-cli/install.sh | sh -
export PATH="$HOME/.gloo/bin:$PATH"
gloo --version

In [ ]:
# gathers namespace + node info for a migration estimate (writes <cluster>.json)
gloo ambient estimate

In [ ]:
# phased plan; pre-reqs is a go/no-go gate, cluster-setup checks ambient-readiness
# before generating recommended-waypoints.yaml / recommended-policies.yaml.
gloo ambient migrate --enterprise --output-dir /tmp/istio-migrate

Field note: on Kubernetes 1.29+ the Solo sidecar is injected as a **native sidecar** (an init container with
`restartPolicy: Always`) and carries `ISTIO_META_ENABLE_HBONE=true`, so it is ambient-ready, but `gloo` v0.2.0's
"sidecars support ambient" check inspects `.spec.containers` and does not see it there. The pre-reqs and
ambient-infrastructure checks are the useful part.

## Migration checklist

**Per cluster, once**
- [ ] Solo images across the mesh, driven by the Gloo Operator's `ServiceMeshController`; Gateway API CRDs present.
- [ ] Flip `dataplaneMode: Ambient`; confirm `istio-cni` + `ztunnel` Ready and sidecars untouched.

**Per namespace, in order**
- [ ] Audit L4 vs L7. Find `PeerAuthentication mode: DISABLE` (no ambient equivalent), `Sidecar`-object egress (moves to egress-waypoint), and any L7 field (needs a waypoint).
- [ ] If L7, deploy the waypoint first and wait for Programmed.
- [ ] Translate selector policies to `targetRefs`; apply the new before removing the old.
- [ ] Bind the workload to the waypoint (`istio.io/use-waypoint`).
- [ ] Enrol: `istio.io/dataplane-mode=ambient`, remove `istio-injection`, delete the old selector L7 policy, rolling-restart.
- [ ] North-south: set `istio.io/ingress-use-waypoint=true` (before any from-waypoint lockdown).
- [ ] Verify routing, L7 authz, L4 allow/deny and a clean load run, from a sidecar caller too.
- [ ] Roll back with a single label if anything is off; clean up old sidecar-era policies once confirmed.

**Optional, once ambient**
- [ ] Move subset canaries to per-version Services + `HTTPRoute`.
- [ ] Retire `Sidecar` egress scoping and `exportTo`.

## Clean up

In [ ]:
kind delete cluster --name ambient-migration